In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
def ler_ultima_particao_tabela_spark(spark, source_table):
  """
  Essa função ler a ultima partição das Tabelas no formato delta baaseado na coluna de data_processamento
  """
  try: 
    # Mais performatica para pegar os metadados
    show_partitions_df = spark.sql(f"SHOW PARTITIONS {source_table}")

    # maior partição da data_processamento
    max_partition = show_partitions_df.agg(f.max("data_processamento")).collect()[0][0]
    print(f"partição maxima {max_partition}")

    # pegar o dataframe com maior partição
    return spark.table(f"{source_table}")\
                      .filter(f.col("data_processamento") == max_partition)
  except Exception as e:
    print(f"Erro ao ler o caminho {source_table}: {e}")
    return None


### 1. CVM FIIs Geral

In [0]:
path_fii_ativo_geral = "workspace.case_spark_cvm.silver_cvm_fii_geral"

df_cvm_fii_geral =  ler_ultima_particao_tabela_spark(spark, path_fii_ativo_geral)

In [0]:
display(df_cvm_fii_geral.orderBy(f.col("data_referencia").desc()))

### 2. Normaliza tipo_fundo_classe

In [0]:
df_cvm_fii_geral = df_cvm_fii_geral.withColumn(
  "tipo_fundo_classe",
  f.when(f.col("tipo_fundo_classe").isNull(), "Fundo")
   .otherwise(f.col("tipo_fundo_classe"))
)

### 3. Criação da Janela e Prioridade

In [0]:
df_cvm_fii_geral = df_cvm_fii_geral.withColumn(
    "prioridade_tipo",
    f.when(f.col("tipo_fundo_classe") == "Classe", 1)
     .otherwise(2)
)

In [0]:
window_mes = Window.partitionBy("cnpj_fundo_classe", "data_referencia").orderBy("prioridade_tipo")

df_cvm_fii_geral = df_cvm_fii_geral.withColumn(
    "rn_mes",
    f.row_number().over(window_mes)
).filter(f.col("rn_mes") == 1)

In [0]:
window_ultimo = Window.partitionBy("cnpj_fundo_classe").orderBy(f.col("data_referencia").desc())

df_cvm_fii_geral = df_cvm_fii_geral\
    .withColumn("rn_ultimo", f.row_number().over(window_ultimo))\
    .filter(f.col("rn_ultimo") == 1)\
    .select(
        "cnpj_fundo_classe",
        "nome_fundo_classe",
        "tipo_fundo_classe",
        "codigo_isin",
        "mandato",
        "segmento_atuacao",
        "tipo_gestao",
        "publico_alvo",
        "prazo_duracao",
        "mercado_negociacao_bolsa",
        "mercado_negociacao_mb",
        "fundo_exclusivo",
        "cotistas_vinculo_familiar",
        "nome_administrador",
        "cnpj_administrador",
        "data_funcionamento",
        "cidade",
        "estado"
    )

### 4. Salvando o Cubo

In [0]:
df_cvm_fii_geral = df_cvm_fii_geral.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_cvm_fii_geral.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.gold_dim_fii")

In [0]:
display(df_cvm_fii_geral)